[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/orange/notebooks/orange_target_selection.ipynb)

# Selecting vulnerable essential targets in M. tuberculosis

**Orange group · Tuberculosis**

*Mycobacterium tuberculosis* has around 4,000 genes and the group needs a shortlist
small enough to study properly. This notebook starts from a genome-wide experiment
that measured, gene by gene, how much each one has to be switched off before the
bacterium stops growing, and narrows it down to the genes that matter in two strains
of *M. tuberculosis* but not in a harmless relative.

## What you will do

- Load three genome-wide knockdown screens: two strains of *M. tuberculosis* and one
  of *M. smegmatis*, a relative that does not cause disease.
- Keep the genes the screen calls essential, treating each screen on its own.
- Filter on how certain the measurement is and on the vulnerability index, and see
  what each rule costs you.
- Keep what survives in both *M. tuberculosis* strains, then remove the genes that are
  just as vulnerable in *M. smegmatis*.
- Look up the UniProt identifier of every target and download the shortlist.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "orange"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. Load the three screens

The data comes from [Bosch et al.,
2021](https://doi.org/10.1016/j.cell.2021.06.033), who used a technique called
**CRISPRi** on every gene in the genome. CRISPRi does not delete a gene; it turns its
volume down. By using many different guides, each of which turns the volume down by a
different amount, the experiment can ask a better question than "can the bacterium
live without this gene?". It asks **how much of the gene the bacterium can afford to
lose** before it stops growing.

That difference matters for drug discovery. A drug almost never shuts a protein off
completely. If a bacterium copes fine until a gene is 90% switched off, a drug would
have to be extraordinarily good to do anything. If the bacterium is already in trouble
at 30%, a much more ordinary drug will work.

The authors ran the experiment three times over: in **H37Rv**, the laboratory strain
almost all TB research uses; in **HN878**, a strain isolated from a patient; and in
***M. smegmatis***, a fast-growing relative that lives in soil and does not cause
disease. All three are in one Excel file, one sheet each.

First, the packages and the file. The Setup cell above put the project folder in place, so the data is already here.

In [ ]:
import os

import pandas as pd
import stylia
from scripts import vulnerability

# Plots: slide format, Ersilia colours
stylia.set_format("slide")
stylia.set_style("ersilia")
nc = stylia.NamedColors()
RANDOM_SEED = 42
DATA = "data/bosc2021_vulnerability.xlsx"

if not os.path.exists(DATA):
    raise FileNotFoundError(f"not found: {DATA}. Run the Setup cell again.")
print(f"reading {DATA}")

Now read the three sheets. Each one becomes a table of its own, and they stay separate for most of this notebook.

In [ ]:
screens = {name: vulnerability.load_screen(DATA, name) for name in vulnerability.SCREENS}
pd.DataFrame([{"screen": name, "sheet": vulnerability.SCREENS[name],
               "genes": len(df), "columns": df.shape[1]}
              for name, df in screens.items()])

*M. smegmatis* has a bigger genome than *M. tuberculosis*, which is why its screen
covers more genes. Here are the first rows of the H37Rv screen, showing only the
columns this notebook uses.

In [ ]:
KEY = ["locus_tag", "name", "crispr_ess", "certain", "vi", "vi_lower", "vi_upper"]
screens["H37Rv"][KEY].head()

## 2. What the columns say about each gene

Each row is one gene. The columns that matter here are:

- **`locus_tag`** is the gene's permanent identifier, like `Rv0667`. Every
  *M. tuberculosis* database uses these, so it is what we will join on later. The
  spreadsheet writes them as `RVBD0667`; `load_screen` has already rewritten them
  into the standard form.
- **`name`** is the gene's common name, like `rpoB`. Many genes do not have one, and
  for those the name is just a copy of the locus tag.
- **`crispr_ess`** is this experiment's verdict: `Essential` or `NonEssential`.
- **`tnseq_ess`** is an older verdict from a different kind of experiment, kept here
  for comparison.
- **`certain`** says whether the experiment collected enough good data about this gene
  to trust the numbers that follow. Section 4 is about this column.
- **`vi`** is the **vulnerability index**, with `vi_lower` and `vi_upper` giving the
  range the true value is 95% likely to lie in. Section 4 is about this too.
- **`antibacterial`** names the drug that already targets this gene, for the handful
  of genes where one exists.

The two essentiality verdicts do not always agree, which is worth seeing before
trusting either of them.

`tnseq_ess` comes from transposon sequencing, which breaks genes completely rather
than turning them down, and it was done years earlier. Comparing the two shows how
much the answer depends on how you ask.

In [ ]:
pd.crosstab(screens["H37Rv"]["crispr_ess"], screens["H37Rv"]["tnseq_ess"])

> **Note:** the older method returns `Uncertain` or `Unknown` for nearly 200 genes,
> while CRISPRi gives a verdict for every one. This notebook follows `crispr_ess`,
> because it is the call that belongs with the vulnerability numbers.

One more thing to notice before filtering. Not every row is a protein-coding gene: the
screen also covers ribosomal RNAs and transfer RNAs, which are made of RNA and never
become proteins. They cannot be drug targets in the sense the group cares about, and
they have no protein identifier, so they will drop out on their own at the very end.
They are easy to spot because their identifier is not an `Rv` number.

In [ ]:
h37 = screens["H37Rv"]
not_genes = h37[~h37["locus_tag"].str.startswith("Rv")]
print(f"{len(not_genes)} of {len(h37):,} rows are not protein-coding genes")
not_genes[["locus_tag", "name", "crispr_ess", "vi"]].head()

## 3. Keep the essential genes, one screen at a time

A target has to be essential: if the bacterium grows perfectly well without the gene,
blocking its protein will not help. So the first filter is `crispr_ess == "Essential"`.

Each screen is filtered on its own, and they are **not** merged yet. The two
*M. tuberculosis* strains are different bacteria grown in different experiments, and
*M. smegmatis* is a different species altogether. Merging them now would hide exactly
the disagreements we want to use later.

In [ ]:
counts = pd.DataFrame([{"screen": name, "genes": len(df),
                        "essential": int((df["crispr_ess"] == "Essential").sum())}
                       for name, df in screens.items()])
counts["share"] = (counts["essential"] / counts["genes"]).map("{:.1%}".format)
counts

Around one gene in six is essential in *M. tuberculosis*, and far fewer in
*M. smegmatis*. That is not a mistake: *M. tuberculosis* has been living inside human
beings for a very long time and has lost many of the genes that let a soil bacterium
fend for itself, so more of what it has left is indispensable.

## 4. How certain the measurement is, and how vulnerable the gene is

Essential is only the beginning. Among essential genes we want the **vulnerable**
ones, and we only want them where the experiment actually measured them well.

**The vulnerability index (`vi`)** summarises how far a gene has to be turned down
before the bacterium suffers. **The more negative the number, the more vulnerable the
gene**, meaning a small reduction is already enough to hurt. Those are the genes worth
making a drug against. In this screen `vi` runs from about -18 to +2.

**`certain`** says whether the experiment had enough guides, spread over enough
different strengths, to fit that curve confidently. When it did not, `vi` is still
reported but its 95% range is enormous, and a number with an enormous range is not a
measurement. Only a small minority of genes qualify.

`add_selection_flags` turns each rule into a True/False column instead of removing rows, so we can keep counting what every rule costs.

In [ ]:
flagged = {name: vulnerability.add_selection_flags(df) for name, df in screens.items()}
h37 = flagged["H37Rv"]
print(f"{h37['is_certain'].sum():,} of {len(h37):,} genes have a confident estimate "
      f"({h37['is_certain'].mean():.0%})")
pd.crosstab(h37["is_essential"], h37["is_certain"])

Essential genes are far more likely to be certain than the rest, which makes sense:
they are the ones where turning the gene down produced a visible effect to measure.

Plotting the vulnerability index of the certain and the uncertain genes side by side
shows why the column cannot be ignored.

In [ ]:
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for certain, color, label in [(False, nc.gray, "uncertain"), (True, nc.orange, "certain")]:
    values = h37.loc[h37["is_certain"] == certain, "vi"]
    ax.hist(values, bins=60, range=(-20, 5), histtype="stepfilled", alpha=0.6,
            color=color, label=f"{label} (n={len(values):,})")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Genes",
             title="Most genes in the screen are not measured confidently")

The uncertain genes are spread all over the scale, including the very vulnerable end.
Some of them may genuinely be vulnerable, but we cannot tell. Plotting each gene's
index against the width of its confidence range makes the difference plain.

In [ ]:
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
spread = h37["vi_upper"] - h37["vi_lower"]
for certain, color, label in [(False, nc.gray, "uncertain"), (True, nc.orange, "certain")]:
    mask = h37["is_certain"] == certain
    ax.scatter(h37.loc[mask, "vi"], spread[mask], color=color, alpha=0.5, s=8,
               label=f"{label} (median width {spread[mask].median():.1f})")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Width of the 95% range",
             title="A certain gene is one with a narrow range")

Now the cutoff. A gene counts as vulnerable when its index is below
**-9.17575**, the value Ersilia used in an earlier *M. tuberculosis* target
prioritisation. There is nothing sacred about it: it is a line drawn on a continuous
scale, and moving it moves the shortlist.

Applying all three rules together, one screen at a time, gives the first real numbers.

In [ ]:
VI_CUTOFF = vulnerability.VI_CUTOFF
steps = pd.DataFrame([{"screen": name,
                       "genes": len(df),
                       "essential": int(df["is_essential"].sum()),
                       "and certain": int((df["is_essential"] & df["is_certain"]).sum()),
                       f"and vi < {VI_CUTOFF:.2f}": int(df["selected"].sum())}
                      for name, df in flagged.items()])
steps

> **Exercise:** the cutoff is a choice, not a fact. Re-run the cell above with
> `VI_CUTOFF = -7` or `VI_CUTOFF = -12` by passing it to
> `vulnerability.add_selection_flags(df, vi_cutoff=...)`, and see how many genes you
> gain or lose. How much would the group's final shortlist change? Is there a number
> of targets you are aiming for, and should that be what decides the cutoff?

## 5. Genes that pass in both M. tuberculosis strains

H37Rv has been grown in laboratories since 1905 and has adapted to that comfortable
life. HN878 was isolated from a patient during an outbreak and belongs to the
W-Beijing family, which is widespread and associated with drug resistance. A target
that only looks good in the laboratory strain is a risk; one that holds up in a
clinical isolate as well is a much safer bet.

So we keep the genes selected in **both**, matching them on their locus tag.

In [ ]:
selected = {name: set(df.loc[df["selected"], "locus_tag"])
            for name, df in flagged.items()}
both = selected["H37Rv"] & selected["HN878"]
print(f"H37Rv {len(selected['H37Rv'])}, HN878 {len(selected['HN878'])}, in both {len(both)}")
print(f"only H37Rv {len(selected['H37Rv'] - selected['HN878'])}, "
      f"only HN878 {len(selected['HN878'] - selected['H37Rv'])}")

Quite a few genes pass in one strain and not the other. Counting them does not say
whether those are real biological differences or genes that simply sat on the wrong
side of the cutoff in one experiment. Plotting one index against the other does.

In [ ]:
pair = flagged["H37Rv"].merge(flagged["HN878"], on="locus_tag", suffixes=("_h37rv", "_hn878"))
in_both = pair["locus_tag"].isin(both)
fig, axs = stylia.create_figure(1, 1, width=0.5, height=0.5)
ax = axs.next()
for mask, color, label in [(~in_both, nc.gray, "not selected"), (in_both, nc.orange, "in both")]:
    ax.scatter(pair.loc[mask, "vi_h37rv"], pair.loc[mask, "vi_hn878"],
               color=color, alpha=0.5, s=8, label=f"{label} ({int(mask.sum()):,})")
ax.axvline(VI_CUTOFF, color=nc.pink, linestyle="--")
ax.axhline(VI_CUTOFF, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="H37Rv vulnerability index", ylabel="HN878 vulnerability index",
             title="The same gene in the laboratory and the clinical strain")

Most of the genes that missed out sit close to the dashed lines, so they are borderline
cases rather than genes that behave completely differently in the two strains. That is
a reason to treat the cutoff gently, and a good argument for looking again at the near
misses if the shortlist turns out too short.

## 6. Remove what is just as vulnerable in M. smegmatis

*M. smegmatis* is a harmless soil-dwelling cousin of *M. tuberculosis*. If a gene is
just as vulnerable there, it is almost certainly a gene every bacterium needs: the
machinery that builds proteins, copies DNA, or makes energy. Those are real
vulnerabilities, but they are not *tuberculosis* vulnerabilities. A drug against one
would likely hit harmless bacteria too, including the ones living in the patient.

Removing them pushes the shortlist towards what is special about the pathogen, which
is what the group's project plan asks for under selectivity.

There is a practical snag. *M. smegmatis* genes have their own identifiers
(`MSMEG0001` and so on), which say nothing about which *M. tuberculosis* gene they
correspond to. What the two screens do share is the common gene name, so that is what
we match on.

In [ ]:
core = flagged["H37Rv"][flagged["H37Rv"]["locus_tag"].isin(both)].copy()
msmeg_selected = set(flagged["Msmeg"].loc[flagged["Msmeg"]["selected"], "name"])
core["in_msmeg"] = core["name"].isin(msmeg_selected)
print(f"{core['in_msmeg'].sum()} of {len(core)} genes are just as vulnerable in M. smegmatis")
core.loc[core["in_msmeg"], ["locus_tag", "name", "vi"]].sort_values("vi")

Look at what came out: `rplB`, `rplF`, `rplP` and `rpsC` are parts of the ribosome,
`rpoA` is part of RNA polymerase, `alaS` and `aspS` load amino acids onto transfer
RNAs, and `tuf` and `secY` are equally fundamental. This is exactly the kind of gene the
comparison is meant to remove, although section 7 shows it does not catch all of
them.

Matching on names has a limit worth stating plainly.

In [ ]:
unnamed = core["name"] == core["locus_tag"]
print(f"{unnamed.sum()} of {len(core)} genes have no common name, so they could not be "
      f"compared with M. smegmatis at all")
core.loc[unnamed, ["locus_tag", "name", "crispr_ess", "vi"]].head()

> **Note:** those genes stay in the shortlist because there is no evidence against
> them, not because they passed a test. Genes without a common name are usually the
> ones nobody has studied, which makes them interesting and risky in equal measure.

That leaves the shortlist.

In [ ]:
targets = core[~core["in_msmeg"]].copy()
print(f"{len(core)} genes in both strains -> {len(targets)} after removing "
      f"the M. smegmatis overlap")

## 7. Look at what came out

Before trusting a shortlist it is worth asking two questions of it: where does it sit
in the data it came from, and does it contain the things we already know to be true?

The first is a picture. The selected genes should be piled up at the vulnerable end of
the scale, well clear of everything that was discarded.

In [ ]:
kept = h37["locus_tag"].isin(targets["locus_tag"])
fig, axs = stylia.create_figure(1, 1)
ax = axs.next()
for mask, color, label in [(~kept, nc.gray, "discarded"), (kept, nc.orange, "selected")]:
    values = h37.loc[mask, "vi"]
    ax.hist(values, bins=60, range=(-20, 5), histtype="stepfilled", alpha=0.6, color=color,
            label=f"{label} (n={len(values):,}, median {values.median():.1f})")
ax.axvline(VI_CUTOFF, color=nc.pink, linestyle="--")
ax.legend()
stylia.label(ax, xlabel="Vulnerability index", ylabel="Genes",
             title="Where the shortlist sits in the whole screen")

The second question is the stronger test. The screen labelled 18 genes with the drug
that already targets them. Those drugs work, so a sensible pipeline ought to rate
those genes as vulnerable. We never used that column to choose anything, so it is a
genuinely independent check.

In [ ]:
known = h37[h37["antibacterial"].notna()].copy()
known["in_shortlist"] = known["locus_tag"].isin(targets["locus_tag"])
print(f"{known['in_shortlist'].sum()} of {len(known)} genes hit by a known TB drug "
      f"are in the shortlist")
known[["locus_tag", "name", "antibacterial", "vi", "is_certain",
       "in_shortlist"]].sort_values("vi")

Four of them come through the whole pipeline: `kasA`, `mmpL3`, `clpC1` and `inhA`, the
target of isoniazid and ethionamide. Three more are vulnerable in H37Rv but fall just
short in HN878: `gyrB` (the fluoroquinolones) at -8.29, `rpoB` (rifampicin) at -8.51
and `dprE1` at -7.84, against a cutoff of -9.18. They were dropped in section 5 for
being borderline, not for behaving differently in the two strains.

The rest are the useful warning. `atpE` is the target of bedaquiline, one of the most
important new TB drugs there is, and this screen does not call it vulnerable at all.
Neither are `embB` (ethambutol) or `alr` (cycloserine).

> **Note:** a low vulnerability index is a good argument for a target, but a high one
> is not proof against it. A filter this strict will always throw away some real
> targets. That is an acceptable price when the aim is a shortlist of strong
> candidates rather than a complete list, but it is worth remembering when a protein
> the group cares about does not appear.

Here is the top of the shortlist itself, most vulnerable first.

In [ ]:
targets.sort_values("vi")[["locus_tag", "name", "vi", "vi_lower", "vi_upper"]].head(10)

The top of that list is almost all ribosomal proteins, which should give you pause.
Section 6 was meant to remove exactly this kind of gene, and it did remove eleven of
them, so why are there still so many?

Because the subtraction can only remove a gene that *M. smegmatis* measured
confidently **and** called vulnerable, and its screen selected just 35 genes against
157 in H37Rv. A gene that every bacterium depends on but that was measured poorly in
*M. smegmatis* survives our filter by default. The filter is only as strong as the
screen behind it.

In [ ]:
ribosomal = targets["name"].str.match(r"^(rpl|rps|rpm)")
print(f"{ribosomal.sum()} of {len(targets)} targets are ribosomal proteins")
targets.loc[~ribosomal].sort_values("vi")[["locus_tag", "name", "vi"]].head(10)

Set the ribosome aside and the rest of the list is more interesting: `glf`, `glfT2`
and `Rv3806c` build the mycobacterial cell wall, which is unlike anything in a human
cell; `acpM`, `kasA` and `accA3` make mycolic acids, the waxy layer that makes
*M. tuberculosis* so hard to kill; `ftsZ` divides the cell. These are the genes the
group's project plan is really asking about.

> **Exercise:** the *M. smegmatis* filter is only as strict as you make it. Try
> subtracting every gene *M. smegmatis* merely calls **essential**, instead of only
> the vulnerable and certain ones, by using `is_essential` in place of `selected` in
> section 6. That removes 41 genes rather than 11 and leaves 56, with 10 ribosomal
> proteins instead of 18. Which shortlist would you rather hand to the group, and what
> have you lost by being stricter?

## 8. Find the UniProt identifier of every target

The group's deliverable is a list of **proteins**, and everything downstream (looking
for a structure, checking whether anyone has made an inhibitor, assessing
druggability) is keyed on a protein identifier rather than a gene one. The standard is
the **UniProt accession**, a code like `P9WGY9`.

Ersilia's earlier prioritisation asked UniProt about one gene at a time and had to
correct about forty-five answers by hand afterwards. The problem was that searching by
name returns *something* for almost any query, and that something is often the wrong
protein or the right protein in the wrong strain.

The fix is to check the answer rather than accept it: only trust an entry when
UniProt's own **ordered locus name** is the `Rv` number we asked about. We do the same
check here, just far more efficiently. Instead of 86 searches we make one request for
every protein in the *M. tuberculosis* proteome and build a table indexed on that
ordered locus name, so the check is the join itself.

In [ ]:
lookup = vulnerability.uniprot_lookup()
print(f"{len(lookup):,} locus tags with a UniProt accession, from one request")
lookup.head(3)

Now attach an accession to each target, and look at whatever fails to match rather than letting it disappear.

In [ ]:
annotated = targets.merge(lookup, on="locus_tag", how="left")
found = annotated["uniprot_ac"].notna()
print(f"{found.sum()} of {len(annotated)} targets have an accession")
annotated.loc[~found, ["locus_tag", "name", "vi"]]

The ones that failed are the ribosomal and transfer RNAs from section 2. They have no
UniProt accession because they are never translated into a protein, so failing to find
one is the correct answer, and they can be dropped.

> **Note:** an empty result is not automatically a mistake. It is worth looking at
> what did not match every time, because the reason can be anything from "this is not
> a protein" to "we joined on the wrong column".

What is left is the group's shortlist of protein targets.

In [ ]:
proteins = annotated[found].sort_values("vi")
COLUMNS = ["locus_tag", "uniprot_ac", "gene_name", "protein_name", "vi", "vi_lower",
           "vi_upper", "reviewed", "antibacterial"]
proteins = proteins[COLUMNS]
print(f"{len(proteins)} protein targets")
proteins.head()

Finally, save it and download it, so the group has the file even after Colab forgets this session.

In [ ]:
os.makedirs("outputs", exist_ok=True)
final_path = "outputs/mtb_selected_targets.csv"
proteins.to_csv(final_path, index=False)
if "google.colab" in sys.modules:
    from google.colab import files
    files.download(final_path)
print(f"{len(proteins)} targets written to {final_path}")

## Summary

- You narrowed 4,052 genes down in four steps: essential (737), measured confidently
  (552), vulnerable (157), and vulnerable in the HN878 clinical strain as well (97).
  Removing the 11 genes that are just as vulnerable in *M. smegmatis* left 86, of
  which 83 are proteins with a UniProt accession.
- The certainty column mattered more than it looks. Only about one gene in seven was
  measured well enough for its vulnerability index to mean anything, and ignoring that
  would have filled the shortlist with noise.
- Of the 18 genes that already have a TB drug against them, 4 came through the whole
  pipeline and 3 more missed only on the clinical strain, which is good evidence the
  selection is sensible. The ones it misses, such as bedaquiline's target `atpE`, are
  a reminder that a high vulnerability index does not rule a target out.
- 18 of the 83 are still ribosomal proteins, because the *M. smegmatis* screen was too
  small to rule them out. The genes worth arguing about are the other 65: the cell
  wall and mycolic acid enzymes, `ftsZ`, and a dozen proteins nobody has named.

**Next:** upload `mtb_selected_targets.csv` to the group's Drive folder so everyone
works from the same list, then go through it together. Which of these proteins has a
known structure, and which of them could a small molecule realistically bind?